# Laboratorio 8 — Validación cruzada y ajuste de hiperparámetros

**Objetivo:** entrenar modelos supervisados de clasificación usando `Pipeline`, `StratifiedKFold`, `GridSearchCV` y `RandomizedSearchCV`.

> Los datasets se entregan completos. El corte train/test se realiza dentro del notebook para evitar fuga de información.

In [ ]:
# Imports + detección automática de la carpeta de datasets
import os, glob, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Ajustar esta ruta en Colab si subes los archivos manualmente.
CANDIDATE_DIRS = ["../datasets/lab08", "./datasets/lab08", "./datasets", "../datasets", "/content/datasets", "/content"]
DATASET_DIR = next((d for d in CANDIDATE_DIRS if os.path.isdir(d) and len(glob.glob(os.path.join(d, "*.csv"))) > 0), None)
if DATASET_DIR is None:
    raise FileNotFoundError("No se encontraron CSV. Sube la carpeta datasets o ajusta DATASET_DIR.")
print("DATASET_DIR =", DATASET_DIR)
print("Archivos:", [os.path.basename(p) for p in glob.glob(os.path.join(DATASET_DIR, "*.csv"))])

In [ ]:
# Funciones reutilizables: cargar CSV, preprocesar, evaluar, separar X/y
def load_csv(name):
    path = os.path.join(DATASET_DIR, name)
    if not os.path.exists(path):
        matches = glob.glob(os.path.join(DATASET_DIR, "*" + name + "*"))
        if matches:
            path = matches[0]
        else:
            raise FileNotFoundError(path)
    return pd.read_csv(path)

def build_preprocessor(X):
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()
    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ])

def evaluate_model(model, X_test, y_test, label="modelo"):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else pred
    out = {
        "modelo": label,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba)
    }
    print(pd.Series(out).to_string())
    print("\nReporte de clasificación")
    print(classification_report(y_test, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.title(f"Matriz de confusión — {label}")
    plt.show()
    return out, pred, proba

def split_xy(df, target_col, drop_id=True):
    y = df[target_col].astype(int)
    X = df.drop(columns=[target_col])
    if drop_id:
        id_like = [c for c in X.columns if c.lower().startswith("id_")]
        X = X.drop(columns=id_like)
    return X, y

## Sección 1 — Baseline con holdout estratificado
Dataset: riesgo académico.

In [ ]:
# Sección 1: baseline con holdout estratificado (LogReg vs RF)
df1 = load_csv("dataset_seccion_1_riesgo_academico.csv")
X, y = split_xy(df1, "target_riesgo_academico")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=80, random_state=RANDOM_STATE, class_weight="balanced")
}

baseline_results = []
for name, clf in models.items():
    pipe = Pipeline(steps=[("prep", build_preprocessor(X_train)), ("clf", clf)])
    pipe.fit(X_train, y_train)
    metrics, _, _ = evaluate_model(pipe, X_test, y_test, name)
    baseline_results.append(metrics)

pd.DataFrame(baseline_results)

## Sección 2 — StratifiedKFold
Dataset: mantenimiento predictivo. Se reporta media y desviación estándar para estimar estabilidad.

In [ ]:
# Sección 2: StratifiedKFold (media y desv. = estabilidad del modelo)
df2 = load_csv("dataset_seccion_2_mantenimiento_maquinas.csv")
X, y = split_xy(df2, "target_falla_30d")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)

pipe = Pipeline(steps=[
    ("prep", build_preprocessor(X_train)),
    ("clf", RandomForestClassifier(n_estimators=80, random_state=RANDOM_STATE, class_weight="balanced"))
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = {"f1":"f1", "roc_auc":"roc_auc", "precision":"precision", "recall":"recall"}
cv_results = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1, return_train_score=False)
summary = {metric.replace("test_", ""): [np.mean(values), np.std(values)] for metric, values in cv_results.items() if metric.startswith("test_")}
pd.DataFrame(summary, index=["media", "desv_est"]).T

## Sección 3 — GridSearchCV
Dataset: fraude transaccional. La clase positiva es poco frecuente, por lo que se usa `class_weight='balanced'` en Random Forest.

In [ ]:
# Sección 3: GridSearchCV (prueba todas las combinaciones de params)
df3 = load_csv("dataset_seccion_3_fraude_transaccional.csv")
X, y = split_xy(df3, "target_fraude")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)

pipe = Pipeline(steps=[
    ("prep", build_preprocessor(X_train)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"))
])

param_grid = {
    "clf__n_estimators": [60, 100],
    "clf__max_depth": [6, None],
    "clf__min_samples_leaf": [2, 5]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipe, param_grid=param_grid, scoring="f1", cv=cv, n_jobs=1, verbose=0)
grid.fit(X_train, y_train)

print("Mejores hiperparámetros:", grid.best_params_)
print("Mejor F1 CV:", grid.best_score_)
metrics_grid, pred_grid, proba_grid = evaluate_model(grid.best_estimator_, X_test, y_test, "RandomForest optimizado")
pd.DataFrame(grid.cv_results_).sort_values("rank_test_score").head(10)[["rank_test_score","mean_test_score","std_test_score","params"]]

## Sección 4 — RandomizedSearchCV y evaluación final
Dataset: calidad de aire. Se selecciona el mejor modelo y se exportan predicciones.

In [ ]:
# Sección 4: RandomizedSearchCV + evaluación final y export de predicciones
df4 = load_csv("dataset_seccion_4_calidad_aire.csv")
X, y = split_xy(df4, "target_alerta_calidad_aire", drop_id=False)
# Eliminamos fecha_hora para este laboratorio, aunque podría convertirse a variables temporales.
if "fecha_hora" in X.columns:
    X = X.drop(columns=["fecha_hora"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)

pipe = Pipeline(steps=[
    ("prep", build_preprocessor(X_train)),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))
])

param_dist = {
    "clf__n_estimators": [60, 100],
    "clf__learning_rate": [0.05, 0.1, 0.15],
    "clf__max_depth": [2, 3],
    "clf__subsample": [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
random_search = RandomizedSearchCV(
    pipe, param_distributions=param_dist, n_iter=4, scoring="f1", cv=cv,
    random_state=RANDOM_STATE, n_jobs=1, verbose=0
)
random_search.fit(X_train, y_train)

print("Mejores hiperparámetros:", random_search.best_params_)
print("Mejor F1 CV:", random_search.best_score_)
metrics_final, pred_final, proba_final = evaluate_model(random_search.best_estimator_, X_test, y_test, "GradientBoosting optimizado")

predicciones = X_test.copy()
predicciones["y_real"] = y_test.values
predicciones["probabilidad_clase_1"] = proba_final
predicciones["prediccion"] = pred_final
predicciones.to_csv("laboratorio_8_predicciones_finales.csv", index=False)

with open("laboratorio_8_metricas_finales.json", "w") as f:
    json.dump(metrics_final, f, indent=2)

print("Archivos exportados: laboratorio_8_predicciones_finales.csv y laboratorio_8_metricas_finales.json")
predicciones.head()

## Actividad final

1. Compare el modelo base con el modelo optimizado.
2. Explique por qué el test solo se usa al final.
3. Indique qué métrica usaría si la clase positiva es crítica.
4. Adjunte el CSV de predicciones en su entrega.

### Respuestas — Actividad final

1. **Modelo base vs modelo optimizado.** El modelo optimizado (con GridSearch/RandomizedSearch) suele mejorar el F1 y el ROC AUC frente al baseline, porque ajusta hiperparámetros como `n_estimators`, `max_depth` o `learning_rate` mediante validación cruzada en lugar de usar valores por defecto. La mejora puede ser pequeña, pero es más robusta porque se valida con varios folds.

2. **Por qué el test solo se usa al final.** Para obtener una estimación honesta del rendimiento. Todo el ajuste (selección de modelo e hiperparámetros) se hace con train + validación cruzada; si usáramos el test en ese proceso, el modelo se "adaptaría" a él (fuga de información) y las métricas finales serían demasiado optimistas. El test debe simular datos nunca vistos.

3. **Qué métrica usar si la clase positiva es crítica.** El **recall** (y el F1 como balance), porque la clase positiva crítica —fraude, falla, riesgo— es la que no queremos dejar pasar. El recall mide qué porcentaje de esos casos reales detectamos; también es útil el ROC AUC para evaluar la separación global. El accuracy no sirve aquí por el desbalance.

4. **CSV de predicciones.** Se genera automáticamente en la Sección 4 como `laboratorio_8_predicciones_finales.csv` (junto con `laboratorio_8_metricas_finales.json`).
